In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

FEATURES = ["HealthIndex", "EngineTemp", "Vibration", "Pressure"]
TARGET = "State"
GROUP = "Equipment"

df = pd.read_csv("digitalTwin.csv")
X = df[FEATURES]
y_text = df[TARGET].astype(str)
groups = df[GROUP]

encoder = LabelEncoder()
y = encoder.fit_transform(y_text)
classes = encoder.classes_

# Séparer par équipement pour éviter d'évaluer sur des équipements déjà vus
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
y_test_text = y_text.iloc[test_idx]

# Modèle XGBoost pour l'évaluation
model_test = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    objective="multi:softprob",
    num_class=len(classes),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
model_test.fit(X_train, y_train)

# Prédictions et métriques sur les équipements gardés de côté
predictions_encoded = model_test.predict(X_test).astype(int)
probabilities = model_test.predict_proba(X_test)
predictions = encoder.inverse_transform(predictions_encoded)

metrics = {
    "accuracy": accuracy_score(y_test, predictions_encoded),
    "balanced_accuracy": balanced_accuracy_score(y_test, predictions_encoded),
    "f1_macro": f1_score(y_test, predictions_encoded, average="macro"),
    "f1_weighted": f1_score(y_test, predictions_encoded, average="weighted"),
}

print("Métriques sur les équipements de test :")
for name, value in metrics.items():
    print(f"{name}: {value:.3f}")

print("\nRapport par état :")
print(classification_report(
    y_test_text,
    predictions,
    labels=classes,
    zero_division=0,
))

print("Matrice de confusion (lignes = réel, colonnes = prédit) :")
print(pd.DataFrame(
    confusion_matrix(y_test_text, predictions, labels=classes),
    index=classes,
    columns=classes,
))

# Comparaison avec un modèle de référence qui prédit toujours l'état majoritaire
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
print("\nAccuracy de référence :", round(accuracy_score(y_test, baseline_predictions), 3))

# Importance des variables
feature_importance = pd.Series(
    model_test.feature_importances_, index=FEATURES
).sort_values(ascending=False)
print("\nImportance des variables :")
print(feature_importance)
feature_importance.to_csv("digital_twin_feature_importance.csv", header=["importance"])

# Exporter les prédictions du jeu de test avec le niveau de confiance
results = df.iloc[test_idx][[GROUP]].copy()
results["State_reel"] = y_test_text.to_numpy()
results["State_predit"] = predictions
results["Confiance"] = probabilities.max(axis=1)
for i, state in enumerate(classes):
    results[f"Probabilite_{state}"] = probabilities[:, i]
results.to_csv("digital_twin_test_predictions.csv", index=False)

model_final = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    objective="multi:softprob",
    num_class=len(classes),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
model_final.fit(X, y)
model_final.save_model("digital_twin_xgboost.json")

print("\nFichiers créés :")
print("- digital_twin_xgboost.json : modèle XGBoost final")
print("- digital_twin_metrics.json : métriques de test")
print("- digital_twin_test_predictions.csv : prédictions et probabilités")
print("- digital_twin_feature_importance.csv : importance des variables")

Métriques sur les équipements de test :
accuracy: 0.331
balanced_accuracy: 0.330
f1_macro: 0.330
f1_weighted: 0.330

Rapport par état :
              precision    recall  f1-score   support

    Critical       0.34      0.31      0.33       679
     Healthy       0.33      0.36      0.35       684
     Warning       0.32      0.31      0.32       670

    accuracy                           0.33      2033
   macro avg       0.33      0.33      0.33      2033
weighted avg       0.33      0.33      0.33      2033

Matrice de confusion (lignes = réel, colonnes = prédit) :
          Critical  Healthy  Warning
Critical       213      245      221
Healthy        204      249      231
Warning        201      259      210

Accuracy de référence : 0.336

Importance des variables :
EngineTemp     0.254216
Vibration      0.249564
HealthIndex    0.249290
Pressure       0.246930
dtype: float32

Fichiers créés :
- digital_twin_xgboost.json : modèle XGBoost final
- digital_twin_metrics.json : métrique